# 03 - Convert NetCDF to Zarr (native and virtual)

Zarr is a generic, self-describing format for chunked, compressed, multi-dimensional arrays. Each chunk is a separate object in the store (or a range within a *shard*), and metadata lives in small JSON documents.

* Pros: the Zarr framework can also be used with non-Zarr files (*virtual* Zarr, see below).
* Cons: features that are well established for rasters (CRS encoding, "implicit" coordinates via affine transformations, overviews) are only slowly making their way into Zarr via the [GeoZarr](https://github.com/zarr-developers/geozarr-spec) specification.

The main design choice is the **chunk layout**: it determines how many requests (and how many bytes) are needed for a given access pattern.

In this notebook we:

1. load the 365 daily NetCDF files from the KNMI gridded mean temperature data in memory (~150 MB);
2. write native Zarr stores with different chunk layouts;
3. create a *virtual* Zarr store ([Icechunk](https://icechunk.io)) that references the original NetCDF files, without copying data.

In [ ]:
import time

import fsspec
import pandas as pd
import xarray as xr

from utils import ENDPOINT_URL, BUCKET, BUCKET_URI, USER_PREFIX_URI, https_url, icechunk_repo

In [ ]:
fs = fsspec.filesystem("s3", endpoint_url=ENDPOINT_URL)

In [ ]:
paths = sorted(fs.glob(f"{BUCKET_URI}/KNMI-Tg1-5/*/*.nc"))
len(paths)

## Load the original files

In [ ]:
ds = xr.open_mfdataset(
    [fs.open(p, mode="rb") for p in paths],
    engine="scipy",
    combine="by_coords",
    data_vars="all",
    parallel="dask",
)
ds

In [ ]:
ds = ds[["prediction"]]
ds.load()

## Native Zarr: chunk layouts

| Layout | Chunk shape (time, y, x) | Optimized for |
|---|---|---|
| `map` | (1, 350, 300) | one day, full domain (as the original files) |
| `timeseries` | (365, 25, 25) | full year at a location |
| `balanced` | (30, 100, 100) | a bit of both |
| `sharded` | chunks (73, 50, 50) in a single (365, 350, 300) shard | small chunks, few objects |

Zarr v3 compresses chunks with Zstandard by default.

In [ ]:
LAYOUTS = {
    "map": dict(chunks=(1, 350, 300)),
    "timeseries": dict(chunks=(365, 25, 25)),
    "balanced": dict(chunks=(30, 100, 100)),
    "sharded": dict(chunks=(73, 50, 50), shards=(365, 350, 300)),
}

In [ ]:
results = []
for name, encoding in LAYOUTS.items():
    store = f"{USER_PREFIX_URI}/KNMI-Tg1-5/zarr/{name}.zarr"
    t0 = time.perf_counter()
    ds.to_zarr(
        store,
        mode="w",
        zarr_format=3,
        consolidated=True,
        encoding={"prediction": encoding},
        storage_options={"endpoint_url": ENDPOINT_URL},
    )
    elapsed = time.perf_counter() - t0
    results.append({
        "layout": name,
        "time_s": round(elapsed, 2),
        "size_MB": round(fs.du(store) / 1e6, 2),
        "objects": len(fs.find(store)),
        **encoding,
    })
pd.DataFrame(results)

> ## Challenge
> * How many chunks need to be read to extract the time series at one location? And a map for a single day?

## Virtual Zarr

NetCDF3 files are uncompressed and store each variable contiguously: every daily file holds one "chunk" of shape (1, 350, 300). [VirtualiZarr](https://virtualizarr.readthedocs.io) scans the files and records the byte ranges of these chunks. The references are then committed to an [Icechunk](https://icechunk.io) repository, which any Zarr reader can open: **cloud-native access without duplicating the data**.

In [ ]:
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store
from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import NetCDF3Parser

registry = ObjectStoreRegistry({BUCKET_URI: S3Store(BUCKET, endpoint=ENDPOINT_URL, skip_signature=True)})
parser = NetCDF3Parser(skip_variables=["iso_dataset", "product", "projection", "stationvalues", "stations"])

vds = open_virtual_mfdataset(
    [f"s3://{p}" for p in paths],
    registry=registry,
    parser=parser,
    combine="by_coords",
    data_vars="all",
    parallel="dask",
    loadable_variables=["x", "y", "time", "lon", "lat"],  # copy small variables in the store
)
vds

NetCDF3 marks missing values with a `_FillValue` attribute (-9999), which the parser moves to the Zarr `fill_value`. Xarray only masks values set in the `_FillValue` attribute, so we add it back (encoded as xarray expects it for Zarr v3).

In [ ]:
import numpy as np
from xarray.backends.zarr import FillValueCoder
vds["prediction"].attrs["_FillValue"] = FillValueCoder.encode(-9999.0, np.dtype("float32"))

In [ ]:
icechunk_store = f"{USER_PREFIX_URI}/KNMI-Tg1-5/icechunk"
if fs.exists(icechunk_store):
    fs.rm(icechunk_store, recursive=True)  # remove if existing
repo = icechunk_repo(icechunk_store)

In [ ]:
session = repo.writable_session("main")
vds.vz.to_icechunk(session.store)
session.commit("Virtual references to the KNMI Tg1 NetCDF files")

In [ ]:
results.append({
    "layout": "virtual",
    "size_MB": round(fs.du(icechunk_store) / 1e6, 2),
    "objects": len(fs.find(icechunk_store)),
    "chunks": (1, 350, 300),
})
pd.DataFrame(results)

Open the virtual store as any Zarr store, and check that it contains the same values as the original files:

In [ ]:
ds_icechunk = xr.open_zarr(repo.readonly_session("main").store)
xr.testing.assert_equal(
    ds_icechunk["prediction"].isel(time=slice(0, 10)),
    ds["prediction"].isel(time=slice(0, 10))
)

Try out the [Gridlook](https://github.com/d70-t/gridlook/) viewer to explore the dataset, it supports Icechunk repositories! Click on the link below:

In [ ]:
print(f"https://gridlook.pages.dev/#{https_url(icechunk_store)}")